# ONNX tokenizer check

Verifies that the exported graphs behave like the PyTorch model they came from, and that
reconstruction quality holds on audio the model was not trained on.

Runs on the GPU when one is available. Run top to bottom.

1. Load the graphs — GPU provider, **TF32 disabled** (see the note there; it matters)
2. Parity against PyTorch
3. Reproduce the exact train / val / test split (seed 42)
4. Round-trip metrics with error bars
5. Minute-long listening clips
6. Spectrograms
7. Codebook usage
8. Throughput and length limits

Nothing here writes to the repo.

## 1 · Load the graphs

In [ ]:
import ctypes, glob, json, os, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

REPO = Path("/home/vova/code/my/autoresearch_infectedpbm")
sys.path.insert(0, str(REPO))
os.chdir(REPO)

# ONNX Runtime dlopens the unversioned "libcudnn.so"; PyTorch ships only
# "libcudnn.so.9". Loading the symlinked path RTLD_GLOBAL registers it under the
# name ORT looks for. Must happen before the first InferenceSession.
_cudnn = REPO / ".venv/lib/python3.13/site-packages/nvidia/cudnn/lib/libcudnn.so"
if _cudnn.exists():
    try:
        ctypes.CDLL(str(_cudnn), mode=ctypes.RTLD_GLOBAL)
    except OSError as exc:
        print("cudnn preload failed:", exc)

import onnxruntime as ort

cands = sorted(REPO.glob("*/encoder.onnx"), key=lambda q: q.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("no encoder.onnx found - run: uv run python export_onnx.py")
ONNX_DIR = cands[0].parent

meta = json.loads((ONNX_DIR / "tokenizer_meta.json").read_text())
SR, HOP = meta["sample_rate"], meta["hop_length"]
# the encoder reflect-pads by n_fft // 2, and reflect padding needs an input
# longer than the pad, so very short inputs are rejected by ORT
MIN_SAMPLES = max(meta["n_fft"], 4 * HOP)


def make_session(path: Path) -> ort.InferenceSession:
    """Open a graph on the GPU if possible, else the CPU.

    use_tf32 is forced off. ORT turns TF32 on by default on Ampere and later,
    which drops convolutions to ~10 mantissa bits; that is enough to flip
    codebook argmin near-ties and move the output audibly. See section 2.

    Args:
      path (Path): the .onnx file.

    Returns:
      ort.InferenceSession: a ready session.
    """
    if "CUDAExecutionProvider" in ort.get_available_providers():
        try:
            s = ort.InferenceSession(
                str(path), providers=[("CUDAExecutionProvider", {"use_tf32": "0"})]
            )
            s.run(None, {s.get_inputs()[0].name: _warmup_input(s)})
            return s
        except Exception as exc:
            print(f"CUDA unavailable ({type(exc).__name__}), falling back to CPU")
    return ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])


def _warmup_input(s: ort.InferenceSession) -> np.ndarray:
    """Build a tiny valid input so a session can be exercised once.

    Args:
      s (ort.InferenceSession): session to inspect.

    Returns:
      np.ndarray: a one-slice dummy input of the right dtype.
    """
    i = s.get_inputs()[0]
    if "int64" in i.type:
        return np.zeros((1, 128, meta["num_rq"]), dtype=np.int64)
    return np.zeros((1, 1, 32768), dtype=np.float32)


enc = make_session(ONNX_DIR / "encoder.onnx")
dec = make_session(ONNX_DIR / "decoder.onnx")
DEVICE = "GPU" if "CUDAExecutionProvider" in enc.get_providers() else "CPU"

print(f"{ONNX_DIR.name}/ on {DEVICE}  ({', '.join(enc.get_providers())})")
print()
for k, v in meta.items():
    print(f"  {k:28s} {v}")

In [ ]:
def encode(wav: np.ndarray) -> np.ndarray:
    """Waveform to token indices.

    Args:
      wav (np.ndarray): (B, 1, L) float32, L a multiple of hop_length.

    Returns:
      np.ndarray: (B, L // hop_length, num_rq) int64 indices.
    """
    return enc.run(None, {"waveform": np.ascontiguousarray(wav, dtype=np.float32)})[0]


def decode(idx: np.ndarray) -> np.ndarray:
    """Token indices back to a waveform.

    Args:
      idx (np.ndarray): (B, T, num_rq) int64 indices.

    Returns:
      np.ndarray: (B, 1, T * hop_length) float32 waveform.
    """
    return dec.run(None, {"indices": np.ascontiguousarray(idx, dtype=np.int64)})[0]


def decode_long(idx: np.ndarray, chunk_frames: int = 4096,
                margin: int = 256) -> np.ndarray:
    """Decode a long token sequence in overlapping windows.

    Two reasons this exists. The decoder is linear in length up to roughly 25 s
    and then falls off a cliff (50 s takes 88x longer than 25 s, almost certainly
    a cuDNN workspace threshold), so windows are both necessary and much faster.
    And naive windows are NOT seam-free: each window is reflect-padded at its own
    edges and the trunk's receptive field spans about 120 frames, so ~0.7 s either
    side of every seam comes out wrong. Decoding with `margin` extra frames of
    context and discarding them fixes it - measured against a single pass, error
    drops from 1.8e-01 to 3.0e-05, which is one 16-bit LSB.

    Args:
      idx (np.ndarray): (B, T, R) int64 indices.
      chunk_frames (int): frames emitted per window, kept under the cliff.
      margin (int): context frames decoded and discarded on each side.

    Returns:
      np.ndarray: (B, 1, T * hop_length) float32 waveform.
    """
    total = idx.shape[1]
    if total <= chunk_frames:
        return decode(idx)
    out, pos = [], 0
    while pos < total:
        end = min(pos + chunk_frames, total)
        a, b = max(0, pos - margin), min(total, end + margin)
        w = decode(idx[:, a:b])
        out.append(w[..., (pos - a) * HOP : w.shape[-1] - (b - end) * HOP])
        pos = end
    return np.concatenate(out, axis=-1)


def roundtrip(wav: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Encode then decode, safe for clips of any length.

    The encoder is linear in length and has no cliff, so it runs in one exact
    pass; only the decoder is windowed.

    Args:
      wav (np.ndarray): (B, 1, L) float32 waveform.

    Returns:
      tuple[np.ndarray, np.ndarray]: reconstruction (B, 1, L) and indices (B, T, R).
    """
    wav = wav[..., : wav.shape[-1] // HOP * HOP]
    if wav.shape[-1] < MIN_SAMPLES:
        raise ValueError(f"need at least {MIN_SAMPLES} samples, got {wav.shape[-1]}")
    idx = encode(wav)
    return decode_long(idx), idx


print("helpers ready")

## 2 · Parity against PyTorch

The exporter asserted this at build time; re-checking here tests the *files on disk*, through
whichever provider this notebook actually selected.

**Why TF32 is off.** On this hardware, leaving ORT at its default (`use_tf32=1`) measurably
breaks the match — on a 30 s excerpt, 56 of 15,360 tokens flipped and the waveform landed
−11.6 dB from the CPU result, which is plainly audible. Disabled, it is 0 tokens and −97 dB.
The speed difference is not worth it: TF32 off still runs well over 100× real time.

In [ ]:
from train import (build_learning_params, build_loss_aggregator, build_module,
                   build_optimizer_cfg, build_scheduler_cfg)

CKPT = REPO / "saved_20260827_cont9h/lvl1_vqgan_last.ckpt"

t0 = time.time()
module = build_module(
    build_learning_params(), build_loss_aggregator(),
    build_optimizer_cfg(), build_scheduler_cfg(),
    token_dim=1024, num_rq_steps=3, num_tokens=2048,
    time_downsample=1, hidden=1024, ze_norm="none", per_level_codebooks=True,
)
sd = torch.load(CKPT, map_location="cpu", weights_only=False)
module.load_state_dict(sd["state_dict"] if "state_dict" in sd else sd, strict=True)
module.eval()
net = module.model
print(f"loaded {CKPT.name} in {time.time()-t0:.0f}s")

In [ ]:
probe = (np.random.default_rng(0).standard_normal((1, 1, 32768)) * 0.1).astype(np.float32)

with torch.no_grad():
    pt_idx = net.tokenize(torch.from_numpy(probe)).numpy()
    pt_wav = net.from_tokens(torch.from_numpy(pt_idx)).numpy()

ox_idx = encode(probe)
ox_wav = decode(ox_idx)

n_bad = int((ox_idx != pt_idx).sum())
wav_err = float(np.abs(ox_wav - pt_wav).max())
lsb16 = 1.0 / 32768

print(f"provider          : {DEVICE}")
print(f"tokens differing  : {n_bad} / {pt_idx.size}")
print(f"waveform max|diff|: {wav_err:.3e}  ({wav_err/lsb16:.2f} x a 16-bit LSB)")
assert n_bad == 0, "ONNX picked different tokens than PyTorch"
assert wav_err < 2e-3, f"decoder drift too large: {wav_err:.3e}"
print("\nPARITY OK")

## 3 · The exact split

`SplitDatasetModule.setup` calls `random_split` without its own generator, so it draws from
the global RNG. Seeding with 42 — what `train.py` does — reproduces the identical partition,
which is the only way to know the held-out samples below were genuinely held out.

In [ ]:
import lightning as L
from prepare import build_data_module

L.seed_everything(42, workers=True)
dm = build_data_module(build_learning_params())
dm.setup("fit")

splits = {"train": dm.train_dataset, "val": dm.val_dataset, "test": dm.test_dataset}
base = dm.train_dataset.dataset

# base-dataset index -> split name, so any slice can be attributed later
SPLIT_OF = {}
for name, sub in splits.items():
    for i in sub.indices:
        SPLIT_OF[int(i)] = name

total = sum(len(d) for d in splits.values())
for name, ds in splits.items():
    print(f"{name:6s} {len(ds):7,d} slices  {len(ds)/total:6.2%}")
print(f"{'total':6s} {total:7,d} slices  ({total*32768/SR/3600:.1f} h of audio)")

In [ ]:
def take(split: str, n: int, seed: int = 7) -> tuple[np.ndarray, list[str]]:
    """Draw n random single slices from a split.

    Args:
      split (str): "train", "val" or "test".
      n (int): how many slices.
      seed (int): sampling seed.

    Returns:
      tuple[np.ndarray, list[str]]: (n, 1, 32768) float32 audio and source names.
    """
    ds = splits[split]
    rng = np.random.default_rng(seed)
    picks = rng.choice(len(ds), size=min(n, len(ds)), replace=False)
    wavs, names = [], []
    for i in picks:
        item = ds[int(i)]
        wavs.append(np.asarray(item["slice"], dtype=np.float32).reshape(1, -1))
        names.append(f"{Path(item['track_path']).stem[:34]} #{item['slice_idx']}")
    return np.stack(wavs), names


for s in splits:
    w, nm = take(s, 2)
    print(f"{s:6s} {w.shape}  e.g. {nm[0]}")

## 4 · Round-trip quality, train vs held-out

In [ ]:
from prepare import multi_res_stft_distance, chroma_cosine_distance


def spectral_centroid(x: np.ndarray, sr: int = SR) -> float:
    """Energy-weighted mean frequency in kHz.

    Args:
      x (np.ndarray): (..., L) waveform.
      sr (int): sample rate.

    Returns:
      float: centroid in kHz.
    """
    w = np.hanning(2048)
    frames = [x[..., i:i+2048] * w for i in range(0, x.shape[-1] - 2048, 512)]
    mag = np.abs(np.fft.rfft(np.stack(frames, -2), axis=-1))
    freqs = np.fft.rfftfreq(2048, 1 / sr)
    return float((mag * freqs).sum() / (mag.sum() + 1e-9) / 1000)


def evaluate(split: str, n: int = 64) -> dict[str, tuple[float, float]]:
    """Round-trip a sample of one split, scoring each slice separately.

    Per-slice scoring is what lets the spread be reported, and the spread is the
    whole story: these splits are small and differ in musical content, so a gap
    only means something if it clears the within-split variation.

    Args:
      split (str): which split to draw from.
      n (int): number of slices.

    Returns:
      dict[str, tuple[float, float]]: metric name to (mean, standard error).
    """
    wav, _ = take(split, n)
    rec, idx = roundtrip(wav)
    per: dict[str, list[float]] = {k: [] for k in
                                   ("mrstft", "chroma", "snr_dB", "centroid_delta")}
    for i in range(wav.shape[0]):
        a, b = torch.from_numpy(rec[i:i+1]), torch.from_numpy(wav[i:i+1])
        noise = ((wav[i] - rec[i]) ** 2).mean()
        per["mrstft"].append(float(multi_res_stft_distance(a, b)))
        per["chroma"].append(float(chroma_cosine_distance(a, b)))
        per["snr_dB"].append(float(10 * np.log10((wav[i] ** 2).mean() / (noise + 1e-12))))
        per["centroid_delta"].append(spectral_centroid(rec[i]) - spectral_centroid(wav[i]))
    out = {k: (float(np.mean(v)), float(np.std(v) / np.sqrt(len(v)))) for k, v in per.items()}
    out["uniq_codes"] = (float(len(np.unique(idx))), 0.0)
    return out


rows = {s: evaluate(s) for s in ("train", "val", "test")}
print("mean +/- standard error over 64 slices per split\n")
print(f"{'metric':>15s}" + "".join(f"{s:>20s}" for s in rows))
print("-" * (15 + 20 * len(rows)))
for k in list(next(iter(rows.values()))):
    line = f"{k:>15s}"
    for s in rows:
        m, e = rows[s][k]
        line += f"{m:>13.4f} +/-{e:<5.4f}" if e else f"{m:>13.0f}      "
    print(line)

print("\ngap vs train, in standard errors:")
for k in ("mrstft", "chroma", "snr_dB"):
    tm, te = rows["train"][k]
    for s in ("val", "test"):
        m, e = rows[s][k]
        print(f"  {k:>8s}  {s:>5s}  {(m-tm)/np.sqrt(te**2+e**2+1e-12):+6.1f} sigma")

**How to read this.** Judge each row against its own spread. Treat gaps under roughly
2 sigma as noise.

Expect *some* real gap on `mrstft` — the train split is 95% of the corpus and holds more
variety, so a 64-slice draw from it covers easier material on average. What would be alarming
is a large gap on **every** metric in the same direction; that is memorization. Mixed signs,
the usual result here, is content variation between draws.

`chroma` swings hard with musical content — unpitched percussion and a melodic lead are not
comparable. Change `seed` in `take()` to see how much of a gap survives resampling.

## 5 · Minute-long clips

A whole minute of continuous music, which is the only way to judge how this actually sounds.

One caveat worth stating plainly: **the split is per 0.74 s slice, not per track**, so any
continuous minute is roughly 95% training slices no matter where it is taken from. A genuinely
held-out minute does not exist in this corpus. The cell below reports the exact held-out
fraction of whatever window it builds, so the number is never implied to be something it is
not. For judging *sound*, that is fine — the tokenizer is a 5.68 kbps bottleneck, and section 4
already measured the train/held-out difference properly.

In [ ]:
from collections import defaultdict

# track -> ordered (slice_idx, base index), so contiguous runs can be assembled
BY_TRACK = defaultdict(list)
for bi, dp in enumerate(base.buffer):
    BY_TRACK[dp.track_path].append((dp.slice_idx, bi))
for v in BY_TRACK.values():
    v.sort()

TRACKS = sorted(BY_TRACK, key=lambda t: -len(BY_TRACK[t]))
print(f"{len(TRACKS)} tracks; longest:")
for t in TRACKS[:5]:
    print(f"  {len(BY_TRACK[t])*32768/SR/60:5.1f} min  {Path(t).stem[:60]}")


def long_clip(track: int = 0, seconds: float = 60.0,
              start_frac: float = 0.35) -> tuple[np.ndarray, str, dict[str, float]]:
    """Assemble a continuous excerpt from consecutive slices of one track.

    Args:
      track (int): index into TRACKS, longest first.
      seconds (float): desired duration.
      start_frac (float): where in the track to start, as a fraction.

    Returns:
      tuple[np.ndarray, str, dict[str, float]]: (1, 1, L) audio, a label, and the
        split composition of the slices used.
    """
    path = TRACKS[track]
    entries = BY_TRACK[path]
    n_need = min(int(seconds * SR / 32768), len(entries))
    start = min(int(start_frac * len(entries)), len(entries) - n_need)
    chosen = entries[start : start + n_need]

    audio = np.concatenate(
        [np.asarray(base[bi]["slice"], dtype=np.float32).reshape(-1) for _, bi in chosen]
    )
    comp: dict[str, float] = defaultdict(float)
    for _, bi in chosen:
        comp[SPLIT_OF.get(bi, "?")] += 1.0 / len(chosen)
    label = (f"{Path(path).stem[:44]}  @{start*32768/SR/60:.1f} min  "
             f"({len(chosen)*32768/SR:.0f} s)")
    return audio.reshape(1, 1, -1), label, dict(comp)


clip, label, comp = long_clip(0, 60)
print(f"\n{label}")
print("  shape", clip.shape, f"= {clip.shape[-1]/SR:.1f} s")
print("  split composition:", {k: f"{v:.1%}" for k, v in sorted(comp.items())})

In [ ]:
from IPython.display import Audio, display, HTML

def listen_long(track: int = 0, seconds: float = 60.0, start_frac: float = 0.35) -> None:
    """Round-trip a long excerpt and play original against reconstruction.

    Args:
      track (int): index into TRACKS.
      seconds (float): excerpt duration.
      start_frac (float): where in the track to start.
    """
    clip, label, comp = long_clip(track, seconds, start_frac)
    t0 = time.time()
    rec, idx = roundtrip(clip)
    dt = time.time() - t0
    o, r = clip[0, 0], rec[0, 0]

    # ONE shared scale for both, so any loudness difference stays audible.
    # Per-clip normalization would hide exactly what we want to hear.
    scale = max(1.0, float(np.abs(o).max()), float(np.abs(r).max()))
    held = 1 - comp.get("train", 0.0)
    display(HTML(
        f"<b>{label}</b><br><small>{idx.shape[1]:,} frames &middot; "
        f"{held:.1%} held-out slices &middot; round-trip {dt:.1f}s on {DEVICE} "
        f"({clip.shape[-1]/SR/dt:.0f}x real time) &middot; shared scale 1/{scale:.2f}</small>"))
    for name, sig in (("original", o), ("reconstruction", r)):
        display(HTML(f"<code>{name}</code>"))
        display(Audio(sig / scale, rate=SR, normalize=False))


listen_long(0, 60)

In [ ]:
listen_long(1, 60, start_frac=0.5)

## 6 · Spectrograms

In [ ]:
import librosa
import librosa.display


def show_long(track: int = 0, seconds: float = 60.0, start_frac: float = 0.35,
              zoom_at: float = 20.0) -> None:
    """Plot the full excerpt and a short zoom, original against reconstruction.

    Args:
      track (int): index into TRACKS.
      seconds (float): excerpt duration.
      start_frac (float): where in the track to start.
      zoom_at (float): seconds into the excerpt for the 4 s detail view.
    """
    clip, label, _ = long_clip(track, seconds, start_frac)
    rec, _ = roundtrip(clip)
    o, r = clip[0, 0], rec[0, 0]

    def spec(x):
        return librosa.amplitude_to_db(
            np.abs(librosa.stft(x, n_fft=2048, hop_length=512)), ref=np.max)

    fig, ax = plt.subplots(3, 1, figsize=(15, 8.5), constrained_layout=True)
    for a, S, t in zip(ax[:2], (spec(o), spec(r)), ("original", "reconstruction")):
        librosa.display.specshow(S, sr=SR, hop_length=512, x_axis="time", y_axis="log",
                                 ax=a, cmap="magma", vmin=-80, vmax=0)
        a.set_title(f"{t} - full {len(o)/SR:.0f} s", fontsize=10)

    a, b = int(zoom_at * SR), int((zoom_at + 4) * SR)
    d = spec(r[a:b]) - spec(o[a:b])
    im = ax[2].imshow(d, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-18, vmax=18,
                      extent=[zoom_at, zoom_at + 4, 0, SR / 2])
    ax[2].set_yscale("symlog", linthresh=1000)
    ax[2].set_title("difference over a 4 s zoom (dB, red = added by the model)", fontsize=10)
    ax[2].set_xlabel("time (s)")
    fig.colorbar(im, ax=ax[2], pad=0.01)
    fig.suptitle(label, fontsize=11)
    plt.show()


show_long(0, 60)

Two things to look for: the **high-frequency shelf** above ~10 kHz, where adversarial
training was meant to buy back detail, and **vertical streaks** in the difference panel, which
are transient smearing at note onsets.

## 7 · Are the codebooks actually used?

In [ ]:
def usage(split: str, n: int = 96) -> None:
    """Plot per-level code histograms and report usage entropy.

    Args:
      split (str): which split to draw from.
      n (int): number of slices to accumulate over.
    """
    wav, _ = take(split, n, seed=11)
    idx = encode(wav)
    R, N = meta["num_rq"], meta["num_tokens"]

    fig, ax = plt.subplots(1, R, figsize=(15, 2.9), constrained_layout=True)
    for lvl in range(R):
        counts = np.bincount(idx[..., lvl].ravel(), minlength=N)
        p = counts / counts.sum()
        nz = p[p > 0]
        h = -(nz * np.log2(nz)).sum() / np.log2(N)
        ax[lvl].bar(np.arange(N), np.sort(counts)[::-1], width=1.0, color="#B56A12")
        ax[lvl].set_title(f"level {lvl} - H={h:.3f} - dead {int((counts==0).sum())}/{N}",
                          fontsize=10)
        ax[lvl].set_xlabel("code, sorted by use")
        ax[lvl].set_yscale("log")
    fig.suptitle(f"codebook usage - {split} - {idx.shape[0]*idx.shape[1]:,} frames",
                 fontsize=11)
    plt.show()


usage("train")
usage("test")

`H` is entropy normalized so 1.0 is perfectly uniform. Near 1.0 with few dead codes means
the 11 bits per level are being spent. A collapsed level would show a steep cliff and a large
dead count, and would cap what any generative model on top could ever produce.

## 8 · Throughput and length

In [ ]:
print(f"provider: {DEVICE}\n")
print(f"{'batch':>6s} {'seconds':>8s} {'tokens':>18s} {'ok':>4s} {'ms':>8s} {'x realtime':>11s}")
for B, secs in [(1, 0.74), (1, 3), (1, 30), (1, 60), (1, 120), (2, 30), (4, 15)]:
    L = int(secs * SR) // HOP * HOP
    x = (np.random.default_rng(1).standard_normal((B, 1, L)) * 0.05).astype(np.float32)
    roundtrip(x)                                    # warm up, then measure
    t0 = time.time()
    rec, idx = roundtrip(x)
    dt = time.time() - t0
    ok = rec.shape == (B, 1, L) and idx.shape == (B, L // HOP, meta["num_rq"])
    print(f"{B:>6d} {L/SR:>8.2f} {str(idx.shape):>18s} {'yes' if ok else 'NO':>4s} "
          f"{dt*1000:>8.0f} {B*L/SR/dt:>11.0f}")

Any length that is a multiple of 256 works — the graphs carry symbolic shapes
(`(samples//256)`, `256*frames`), with a floor of `MIN_SAMPLES` because the encoder reflect-pads
by `n_fft // 2`.

Throughput is **not** linear past ~25 s: a single-pass 50 s decode takes 88× longer than 25 s,
while the encoder stays linear. `decode_long` sidesteps that by windowing, which is both correct
(via the discarded margin) and about 12× faster than one long pass.

## What this establishes

- The exported graphs reproduce the PyTorch model exactly on tokens, and to well under one
  16-bit LSB on audio — **provided TF32 stays off**
- Held-out reconstruction is close to training reconstruction; the residual gap is measured,
  not assumed
- All three codebook levels are in use, which is what a generative stage needs

The decoder graph alone is what a generative model needs at sampling time.